In [ ]:
"""
MCP Server for Note Management

This module implements an MCP server that manages notes with priorities
and categories.

The server exposes:
- Tools for creating, retrieving, listing, updating, and deleting notes.
- Resources for accessing all notes or filtering notes by priority/category.
"""

import json

from mcp.server import Server
from mcp.types import (
    CallToolResult,
    ListResourcesResult,
    ListToolsResult,
    ReadResourceResult,
    Resource,
    TextContent,
    TextResourceContents,
    Tool,
)

from storage import NoteStorage
from tool_handlers import (
    _storage,
    create_note_handler,
    delete_note_handler,
    get_note_handler,
    list_notes_handler,
    update_note_handler,
)


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

PRIORITIES = ["low", "medium", "high"]
CATEGORIES = ["work", "personal", "ideas"]


# ---------------------------------------------------------------------------
# Storage
# ---------------------------------------------------------------------------

# IMPORTANT:
# The tool handlers already use their own NoteStorage instance.
# We intentionally reuse that same instance here so that:
#
#   tools -> storage
#   resources -> same storage
#
# remain consistent.
#
# This also allows the tests to reset mcp_server._storage.clear().
storage: NoteStorage = _storage


# Backwards-compatible name used by the tests.
_storage = storage


# ---------------------------------------------------------------------------
# Tool definitions
# ---------------------------------------------------------------------------

TOOLS = [
    Tool(
        name="create_note",
        description=(
            "Creates a new note with the given title, content, priority, and category."
        ),
        input_schema={
            "type": "object",
            "properties": {
                "title": {
                    "type": "string",
                    "description": "The title of the note.",
                },
                "content": {
                    "type": "string",
                    "description": "The content of the note.",
                },
                "priority": {
                    "type": "string",
                    "enum": PRIORITIES,
                    "description": "The priority level of the note.",
                },
                "category": {
                    "type": "string",
                    "enum": CATEGORIES,
                    "description": "The category of the note.",
                },
            },
            "required": [
                "title",
                "content",
                "priority",
                "category",
            ],
            "additionalProperties": False,
        },
    ),
    Tool(
        name="get_note",
        description="Gets a note by its ID.",
        input_schema={
            "type": "object",
            "properties": {
                "note_id": {
                    "type": "string",
                    "description": "The ID of the note to retrieve.",
                }
            },
            "required": ["note_id"],
            "additionalProperties": False,
        },
    ),
    Tool(
        name="list_notes",
        description=(
            "Lists all notes, optionally filtered by priority and/or category."
        ),
        input_schema={
            "type": "object",
            "properties": {
                "priority": {
                    "type": "string",
                    "enum": ["all", *PRIORITIES],
                    "description": "Filter notes by priority.",
                },
                "category": {
                    "type": "string",
                    "enum": ["all", *CATEGORIES],
                    "description": "Filter notes by category.",
                },
            },
            "additionalProperties": False,
        },
    ),
    Tool(
        name="update_note",
        description="Updates an existing note with the given ID.",
        input_schema={
            "type": "object",
            "properties": {
                "note_id": {
                    "type": "string",
                    "description": "The ID of the note to update.",
                },
                "updates": {
                    "type": "object",
                    "description": "The fields to update.",
                    "properties": {
                        "title": {
                            "type": "string",
                            "description": "The new title.",
                        },
                        "content": {
                            "type": "string",
                            "description": "The new content.",
                        },
                        "priority": {
                            "type": "string",
                            "enum": PRIORITIES,
                            "description": "The new priority.",
                        },
                        "category": {
                            "type": "string",
                            "enum": CATEGORIES,
                            "description": "The new category.",
                        },
                    },
                    "minProperties": 1,
                    "additionalProperties": False,
                },
            },
            "required": [
                "note_id",
                "updates",
            ],
            "additionalProperties": False,
        },
    ),
    Tool(
        name="delete_note",
        description="Deletes a note by its ID.",
        input_schema={
            "type": "object",
            "properties": {
                "note_id": {
                    "type": "string",
                    "description": "The ID of the note to delete.",
                }
            },
            "required": ["note_id"],
            "additionalProperties": False,
        },
    ),
]


# ---------------------------------------------------------------------------
# Tool handlers
# ---------------------------------------------------------------------------


async def handle_list_tools(ctx, params) -> ListToolsResult:
    """Return all available note-management tools."""
    return ListToolsResult(tools=TOOLS)


async def handle_call_tool(ctx, params) -> CallToolResult:
    """Route an MCP tool call to the corresponding tool handler."""

    try:
        arguments = params.arguments or {}

        if params.name == "create_note":
            result = create_note_handler(
                arguments["title"],
                arguments["content"],
                arguments["priority"],
                arguments["category"],
            )

        elif params.name == "get_note":
            result = get_note_handler(
                arguments["note_id"],
            )

        elif params.name == "list_notes":
            result = list_notes_handler(
                arguments.get("priority", "all"),
                arguments.get("category", "all"),
            )

        elif params.name == "update_note":
            result = update_note_handler(
                arguments["note_id"],
                arguments["updates"],
            )

        elif params.name == "delete_note":
            result = delete_note_handler(
                arguments["note_id"],
            )

        else:
            return CallToolResult(
                is_error=True,
                content=[
                    TextContent(
                        type="text",
                        text=json.dumps(
                            {"error": f"Unknown tool: {params.name}"},
                            ensure_ascii=False,
                        ),
                    )
                ],
            )

        try:
            parsed_result = json.loads(result)
        except (TypeError, json.JSONDecodeError):
            parsed_result = result

        is_error = (
            isinstance(parsed_result, dict)
            and "error" in parsed_result
            and parsed_result.get("success") is not True
        )

        return CallToolResult(
            is_error=is_error,
            content=[
                TextContent(
                    type="text",
                    text=(
                        json.dumps(
                            parsed_result,
                            ensure_ascii=False,
                            default=str,
                        )
                        if not isinstance(parsed_result, str)
                        else parsed_result
                    ),
                )
            ],
        )

    except Exception as exc:
        return CallToolResult(
            is_error=True,
            content=[
                TextContent(
                    type="text",
                    text=json.dumps(
                        {"error": str(exc)},
                        ensure_ascii=False,
                    ),
                )
            ],
        )


# ---------------------------------------------------------------------------
# Resource definitions
# ---------------------------------------------------------------------------

RESOURCES = [
    Resource(
        uri="notes://all",
        name="All Notes",
        description="All notes.",
        mime_type="application/json",
    ),
]

RESOURCES.extend(
    Resource(
        uri=f"notes://priority/{priority}",
        name=f"{priority.capitalize()} Priority Notes",
        description=f"Notes with {priority} priority.",
        mime_type="application/json",
    )
    for priority in PRIORITIES
)

RESOURCES.extend(
    Resource(
        uri=f"notes://category/{category}",
        name=f"{category.capitalize()} Notes",
        description=f"Notes in the {category} category.",
        mime_type="application/json",
    )
    for category in CATEGORIES
)


# ---------------------------------------------------------------------------
# Resource handlers
# ---------------------------------------------------------------------------


async def handle_list_resources(ctx, params) -> ListResourcesResult:
    """Return all available note resources."""
    return ListResourcesResult(
        resources=RESOURCES,
    )


async def handle_read_resource(ctx, params) -> ReadResourceResult:
    """Read notes from a resource URI."""

    uri = str(params.uri)

    if uri == "notes://all":
        notes = _storage.list_notes()

    elif uri.startswith("notes://priority/"):
        priority = uri.removeprefix("notes://priority/")

        if priority not in PRIORITIES:
            raise ValueError(f"Invalid priority: {priority}")

        notes = _storage.list_notes(
            priority=priority,
        )

    elif uri.startswith("notes://category/"):
        category = uri.removeprefix("notes://category/")

        if category not in CATEGORIES:
            raise ValueError(f"Invalid category: {category}")

        notes = _storage.list_notes(
            category=category,
        )

    else:
        raise ValueError(f"Invalid resource URI: {uri}")

    return ReadResourceResult(
        contents=[
            TextResourceContents(
                uri=uri,
                mime_type="application/json",
                text=json.dumps(
                    notes,
                    ensure_ascii=False,
                    default=str,
                ),
            )
        ]
    )


# ---------------------------------------------------------------------------
# MCP Server
# ---------------------------------------------------------------------------

server = Server(
    "note-management-server",
    on_list_tools=handle_list_tools,
    on_call_tool=handle_call_tool,
    on_list_resources=handle_list_resources,
    on_read_resource=handle_read_resource,
)


In [ ]:
"""
Tool handler implementations for the MCP server.

This module provides the actual implementations of tool functions.
No changes needed - this is provided as working code.
"""

from typing import Dict, List
import json
from storage import NoteStorage

# Global storage instance
_storage = NoteStorage()


def create_note_handler(title: str, content: str, priority: str, category: str) -> str:
    """
    Handler for create_note tool.

    Args:
        title: Note title
        content: Note content
        priority: Priority level
        category: Note category

    Returns:
        JSON string with created note data
    """
    note = _storage.create_note(title, content, priority, category)
    return json.dumps(note, indent=2)


def get_note_handler(note_id: str) -> str:
    """
    Handler for get_note tool.

    Args:
        note_id: Note ID to retrieve

    Returns:
        JSON string with note data or error message
    """
    note = _storage.get_note(note_id)
    if note:
        return json.dumps(note, indent=2)
    else:
        return json.dumps({"error": f"Note not found: {note_id}"}, indent=2)


def list_notes_handler(priority: str = "all", category: str = "all") -> str:
    """
    Handler for list_notes tool.

    Args:
        priority: Filter by priority or "all"
        category: Filter by category or "all"

    Returns:
        JSON string with list of notes
    """
    priority_filter = None if priority == "all" else priority
    category_filter = None if category == "all" else category

    notes = _storage.list_notes(priority=priority_filter, category=category_filter)
    return json.dumps(notes, indent=2)


def update_note_handler(note_id: str, updates: Dict) -> str:
    """
    Handler for update_note tool.

    Args:
        note_id: Note ID to update
        updates: Dictionary of fields to update

    Returns:
        JSON string with updated note data or error message
    """
    note = _storage.update_note(note_id, updates)
    if note:
        return json.dumps(note, indent=2)
    else:
        return json.dumps({"error": f"Note not found: {note_id}"}, indent=2)


def delete_note_handler(note_id: str) -> str:
    """
    Handler for delete_note tool.

    Args:
        note_id: Note ID to delete

    Returns:
        JSON string with success status
    """
    success = _storage.delete_note(note_id)
    if success:
        return json.dumps({"success": True, "message": f"Note {note_id} deleted successfully"}, indent=2)
    else:
        return json.dumps({"success": False, "error": f"Note not found: {note_id}"}, indent=2)


In [ ]:
"""
Simple in-memory storage for notes.

This module provides a basic storage layer for the MCP server.
No changes needed - this is provided as working code.
"""

from typing import Dict, List, Optional
import uuid
from datetime import datetime


class NoteStorage:
    """In-memory storage for notes"""

    def __init__(self):
        """Initialize empty storage"""
        self._notes: Dict[str, Dict] = {}

    def create_note(self, title: str, content: str, priority: str, category: str) -> Dict:
        """
        Create a new note.

        Args:
            title: Note title
            content: Note content
            priority: Priority level ("low", "medium", "high")
            category: Note category ("work", "personal", "ideas")

        Returns:
            Dictionary with note data including generated ID
        """
        note_id = f"note_{uuid.uuid4().hex[:8]}"
        note = {
            "id": note_id,
            "title": title,
            "content": content,
            "priority": priority,
            "category": category,
            "created_at": datetime.now().isoformat(),
            "updated_at": datetime.now().isoformat()
        }
        self._notes[note_id] = note
        return note

    def get_note(self, note_id: str) -> Optional[Dict]:
        """
        Get a note by ID.

        Args:
            note_id: Note ID to retrieve

        Returns:
            Note dictionary if found, None otherwise
        """
        return self._notes.get(note_id)

    def list_notes(self, priority: Optional[str] = None, category: Optional[str] = None) -> List[Dict]:
        """
        List all notes, optionally filtered by priority and/or category.

        Args:
            priority: Filter by priority ("low", "medium", "high") or None for all
            category: Filter by category ("work", "personal", "ideas") or None for all

        Returns:
            List of note dictionaries
        """
        notes = list(self._notes.values())

        if priority:
            notes = [n for n in notes if n["priority"] == priority]

        if category:
            notes = [n for n in notes if n["category"] == category]

        return sorted(notes, key=lambda x: x["created_at"], reverse=True)

    def update_note(self, note_id: str, updates: Dict) -> Optional[Dict]:
        """
        Update an existing note.

        Args:
            note_id: Note ID to update
            updates: Dictionary of fields to update

        Returns:
            Updated note dictionary if found, None otherwise
        """
        if note_id not in self._notes:
            return None

        note = self._notes[note_id]
        note.update(updates)
        note["updated_at"] = datetime.now().isoformat()
        return note

    def delete_note(self, note_id: str) -> bool:
        """
        Delete a note.

        Args:
            note_id: Note ID to delete

        Returns:
            True if deleted, False if not found
        """
        if note_id in self._notes:
            del self._notes[note_id]
            return True
        return False

    def get_all_notes(self) -> List[Dict]:
        """Get all notes"""
        return list(self._notes.values())

    def clear(self):
        """Clear all notes (for testing)"""
        self._notes.clear()
